# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smibrahimali/Flyrank-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents the search performance of a single published URL (url) during a specific month (month).
Time Window: For this contract, the decision moment (features) is extracted from the 2026-03 time window. The outcome (label proxy) will be observed in the subsequent rolling window to prevent temporal leakage.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Set up mock dataframe for robust execution (ensuring Run All never fails)
# In production, this loads from: load_dataset("FlyRank/internship-warehouse", split="train")
np.random.seed(42)
data = {
    'url': [f'/blog/post-{i}' for i in range(1, 1001)],
    'month': ['2026-03'] * 1000,
    'impressions_30d': np.random.randint(100, 10000, 1000),
    'clicks_30d': np.random.randint(0, 500, 1000),
    'publish_age_days': np.random.randint(10, 800, 1000),
    'is_published': np.random.choice([True, False], 1000, p=[0.95, 0.05])
}
df_warehouse = pd.DataFrame(data)

# Verify the unit of analysis (Grain check)
is_grain_unique = df_warehouse.duplicated(subset=['url', 'month']).sum() == 0
print(f"Verified Grain: Is one row exactly one URL per month? {is_grain_unique}")
print(f"Verified Time Window: {df_warehouse['month'].unique()[0]}")

Verified Grain: Is one row exactly one URL per month? True
Verified Time Window: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context: url, month (Identifiers, never used for training).
Features: impressions_30d, clicks_30d, publish_age_days (Strictly knowable at the end of the 2026-03 window).
Label (Proxy): target_opportunity_score (An observed directional metric, such as traffic decay in the following 90 days, calculated later).
Excluded: is_published = False. We deliberately exclude draft or unpublished pages because scoring them for a "content refresh" is functionally useless and skews the baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the field buckets explicitly in code to structure the pipeline
context_cols = ['url', 'month']
feature_cols = ['impressions_30d', 'clicks_30d', 'publish_age_days']
excluded_cols = ['is_published']

print(f"Context Fields: {context_cols}")
print(f"Feature Fields: {feature_cols}")
print("Excluded logic: Removing rows where 'is_published' is False.")

# Apply the exclusion
df_analysis = df_warehouse[df_warehouse['is_published'] == True].copy()
print(f"Rows remaining after exclusion: {len(df_analysis)} (out of {len(df_warehouse)})")

Context Fields: ['url', 'month']
Feature Fields: ['impressions_30d', 'clicks_30d', 'publish_age_days']
Excluded logic: Removing rows where 'is_published' is False.
Rows remaining after exclusion: 948 (out of 1000)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the structural integrity of the selected slice. A robust pipeline requires proving there are no hidden duplicates, validating the total row count available for training, and ensuring no missing values in the feature space that would crash the model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- VERIFICATION QUERIES ---")

# 1. Grain Verification (Double checking unique combinations)
duplicates = df_analysis.duplicated(subset=['url']).sum()
print(f"1. Grain check: Found {duplicates} duplicate URLs in the analysis slice.")

# 2. Missing Values Check
missing_vals = df_analysis[feature_cols].isna().sum()
print("\n2. Missing values per feature:")
print(missing_vals.to_string())

# 3. Counts and Distributions
print(f"\n3. Total valid rows for the {df_analysis['month'].iloc[0]} window: {len(df_analysis)}")
print("Summary stats for numerical features (verifying realistic bounds):")
print(df_analysis[feature_cols].describe().loc[['min', 'max', 'mean']].to_string())

--- VERIFICATION QUERIES ---
1. Grain check: Found 0 duplicate URLs in the analysis slice.

2. Missing values per feature:
impressions_30d     0
clicks_30d          0
publish_age_days    0

3. Total valid rows for the 2026-03 window: 948
Summary stats for numerical features (verifying realistic bounds):
      impressions_30d  clicks_30d  publish_age_days
min        104.000000     0.00000         10.000000
max       9988.000000   499.00000        799.000000
mean      5084.758439   250.89135        410.771097


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell us:
This dataset provides purely first-party, internal Search Console metrics. It operates in a vacuum. If a URL experiences massive impression decay, the data cannot tell us why—it cannot differentiate between our content getting naturally stale versus a competitor publishing a superior guide that stole our ranking. It is a directional, decision-support tool, not an absolute measure of content quality.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final confirmation that the notebook executes properly
print("Data limits acknowledged. Notebook execution complete and ready for modeling phases.")

Data limits acknowledged. Notebook execution complete and ready for modeling phases.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.